# Previsão de Corridas

Exploração do banco de dados SQLite da Fórmula 1: visão geral do schema, qualidade dos dados e análise.

**Objetivo do projeto:** criar um modelo capaz de prever o *standing* (classificação) da corrida antes mesmo dela começar.

**Conteúdo**
1. Inicialização (imports, configuração, helpers e conexão)
2. Visão geral do schema
3. Qualidade dos dados (relatório de nulos)
4. Limpeza das corridas sprint
5. Exploração de `race_data`

## 1. Inicialização

Um único bloco que prepara tudo para o resto do notebook rodar: imports,
configuração, helpers e conexão com a base de trabalho.

Como funciona a base de dados:

- `DB_ORIGINAL` (`formula.db`): o arquivo baixado, NUNCA é modificado.
- `DB_TRABALHO` (`formula_trabalho.db`): cópia usada em todas as consultas e alterações. Precisa já existir: crie-a uma vez copiando o original.

Helpers disponíveis:

- `connect()`: abre (ou reutiliza) a conexão com a cópia de trabalho, usando `sqlite3.Row` como row factory. Exige que a cópia já exista.
- `query(sql, params)`: executa um SELECT e retorna um `DataFrame` (o helper principal).
- `run(sql, params)`: executa comandos não-SELECT (DDL/DML) e retorna o número de linhas afetadas.
- `row_count(table)`: retorna o número de linhas de uma tabela.
- `box_plot(...)`: box plot de `numcol` agrupado por `catcol`, a partir de SQL ou de um DataFrame.

In [14]:
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Configuração dos bancos
DB_ORIGINAL = Path("formula.db")
DB_TRABALHO = Path("formula_trabalho.db")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 6)

# Helpers de consulta
_conn: sqlite3.Connection | None = None


def connect() -> sqlite3.Connection:
    """Abre (ou reutiliza) a conexão com a cópia de trabalho do banco.

    A cópia (DB_TRABALHO) precisa já existir; o original nunca é aberto
    nem modificado.
    """
    global _conn
    if _conn is None:
        if not DB_TRABALHO.exists():
            raise FileNotFoundError(
                f"Cópia de trabalho '{DB_TRABALHO}' não encontrada. "
                f"Crie-a a partir de '{DB_ORIGINAL}' antes de continuar."
            )
        _conn = sqlite3.connect(DB_TRABALHO)
        _conn.row_factory = sqlite3.Row
        print(f"Banco de dados '{DB_TRABALHO}' aberto com sucesso.")
    return _conn


def query(sql: str, params: tuple = ()) -> pd.DataFrame:
    """Executa um SELECT e retorna o resultado como DataFrame."""
    return pd.read_sql_query(sql, connect(), params=params)


def run(sql: str, params: tuple = ()) -> int:
    """Executa um comando DDL/DML e retorna o número de linhas afetadas."""
    cur = connect().execute(sql, params)
    _conn.commit()
    return cur.rowcount


def row_count(table: str) -> int:
    """Retorna o número de linhas de uma tabela."""
    return query(f"SELECT COUNT(*) AS n FROM {table}")["n"].iloc[0]


# Helpers de visualização
def box_plot(query_or_df, catcol: str, numcol: str, **kwargs):
    """Box plot de numcol agrupado por catcol, a partir de SQL ou de um DataFrame."""
    df = query(query_or_df) if isinstance(query_or_df, str) else query_or_df
    plt.figure(figsize=(8, 6))
    ax = sns.boxplot(x=catcol, y=numcol, data=df, **kwargs)
    plt.tight_layout()
    return ax


# Inicializa a conexão
conn = connect()

Banco de dados 'formula_trabalho.db' aberto com sucesso.


## 2. Visão geral do schema

Helpers para inspecionar tabelas, colunas e contagem de linhas.

In [15]:
def list_tables() -> list[str]:
    """Retorna os nomes de todas as tabelas do banco, em ordem alfabética."""
    rows = connect().execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
    return [row[0] for row in rows]


def columns(table: str) -> pd.DataFrame:
    """Retorna o PRAGMA table_info de uma tabela como DataFrame."""
    return query(f"PRAGMA table_info({table})")


def schema_overview() -> pd.DataFrame:
    """Uma linha por tabela: quantidade de colunas e de linhas."""
    tabelas = list_tables()
    return pd.DataFrame(
        [{"table": t, "columns": len(columns(t)), "rows": row_count(t)} for t in tabelas]
    )

In [16]:
tabelas = list_tables()
print(f"Tabelas disponíveis no banco ({len(tabelas)}):")
for i, nome_tabela in enumerate(tabelas, start=1):
    print(f"{i:2}. {nome_tabela}")

Tabelas disponíveis no banco (31):
 1. chassis
 2. circuit
 3. circuit_layout
 4. constructor
 5. constructor_chronology
 6. continent
 7. country
 8. driver
 9. driver_family_relationship
10. engine
11. engine_manufacturer
12. entrant
13. grand_prix
14. race
15. race_constructor_standing
16. race_data
17. race_driver_standing
18. season
19. season_constructor
20. season_constructor_standing
21. season_driver
22. season_driver_standing
23. season_engine_manufacturer
24. season_entrant
25. season_entrant_chassis
26. season_entrant_constructor
27. season_entrant_driver
28. season_entrant_engine
29. season_entrant_tyre_manufacturer
30. season_tyre_manufacturer
31. tyre_manufacturer


In [17]:
schema_overview()

,table,columns,rows
0,chassis,4,1153
1,circuit,13,78
2,circuit_layout,5,160
3,constructor,22,187
4,constructor_chronology,5,217
5,continent,4,7
6,country,7,249
7,driver,32,917
8,driver_family_relationship,4,86
9,engine,7,424


## 3. Qualidade dos dados (relatório de nulos)

`null_report()` calcula o percentual de nulos de cada coluna de uma tabela.
Sem argumento, gera o relatório do banco inteiro. Entender onde há dados faltantes
é essencial para decidir quais *features* podem alimentar o modelo de previsão.

In [18]:
def null_report(table: str) -> pd.DataFrame:
    """Retorna contagem e percentual de nulos por coluna de uma tabela."""
    total = row_count(table)
    if total == 0:
        return pd.DataFrame(columns=["table", "column", "nulls", "rows", "null_pct"])

    colunas = [row[1] for row in connect().execute(f"PRAGMA table_info({table})")]
    contagens = {
        col: connect().execute(f"SELECT COUNT(*) FROM {table} WHERE {col} IS NULL").fetchone()[0]
        for col in colunas
    }
    return pd.DataFrame(
        [
            {
                "table": table,
                "column": col,
                "nulls": n,
                "rows": total,
                "null_pct": round(100 * n / total, 2),
            }
            for col, n in contagens.items()
        ]
    )


def full_null_report(tables: list[str] | None = None) -> pd.DataFrame:
    """Relatório de nulos de todas as tabelas (ou apenas das informadas)."""
    tabelas = tables or list_tables()
    return pd.concat([null_report(t) for t in tabelas], ignore_index=True)

In [19]:
report = full_null_report()

print("Percentual de nulos por coluna em cada tabela:")
print("-" * 50)
for nome_tabela, grupo in report.groupby("table", sort=False):
    print(f"\nTabela: {nome_tabela}")
    for _, linha in grupo.iterrows():
        print(f"  - {linha.column}: {linha.null_pct:.2f}% nulos ({linha.nulls}/{linha.rows})")

Percentual de nulos por coluna em cada tabela:
--------------------------------------------------

Tabela: chassis
  - id: 0.00% nulos (0/1153)
  - constructor_id: 0.00% nulos (0/1153)
  - name: 0.00% nulos (0/1153)
  - full_name: 0.00% nulos (0/1153)

Tabela: circuit
  - id: 0.00% nulos (0/78)
  - name: 0.00% nulos (0/78)
  - full_name: 0.00% nulos (0/78)
  - previous_names: 79.49% nulos (62/78)
  - type: 0.00% nulos (0/78)
  - direction: 0.00% nulos (0/78)
  - place_name: 0.00% nulos (0/78)
  - country_id: 0.00% nulos (0/78)
  - latitude: 0.00% nulos (0/78)
  - longitude: 0.00% nulos (0/78)
  - length: 0.00% nulos (0/78)
  - turns: 0.00% nulos (0/78)
  - total_races_held: 0.00% nulos (0/78)

Tabela: circuit_layout
  - id: 0.00% nulos (0/160)
  - circuit_id: 0.00% nulos (0/160)
  - effective: 0.00% nulos (0/160)
  - length: 0.00% nulos (0/160)
  - turns: 0.00% nulos (0/160)

Tabela: constructor
  - id: 0.00% nulos (0/187)
  - name: 0.00% nulos (0/187)
  - full_name: 0.00% nulos (0/187

In [20]:
# Visão compacta: apenas colunas com nulos, ordenadas por severidade
report[report.null_pct > 0].sort_values("null_pct", ascending=False).head(25)

,table,column,nulls,rows,null_pct
163,race,qualifying_1_date,1172,1172,100.00
178,race,warming_up_time,1172,1172,100.00
177,race,warming_up_date,1172,1172,100.00
153,race,pre_qualifying_date,1172,1172,100.00
154,race,pre_qualifying_time,1172,1172,100.00
166,race,qualifying_2_time,1172,1172,100.00
165,race,qualifying_2_date,1172,1172,100.00
164,race,qualifying_1_time,1172,1172,100.00
161,race,free_practice_4_date,1172,1172,100.00
162,race,free_practice_4_time,1172,1172,100.00


### O caso contra as corridas sprint

Antes de remover as sprints, vamos embasar a decisão nos dados:

- **Formato novo**: a sprint foi introduzida em 2021, então só existe numa fração recente do histórico.
- **Outro ruleset**: a sprint tem pontuação, distância e classificação próprias, ou seja, não é diretamente comparável com a corrida principal que queremos prever.
- **Nulos demais**: no geral, as colunas de sprint ficam vazias para quase todas as corridas do banco, como o relatório acima mostra.

As evidências abaixo consultam o ORIGINAL em modo somente leitura (`mode=ro`, sem risco
de alterá-lo), para que continuem válidas mesmo depois de a limpeza já ter rodado.

In [21]:
# Conexão somente leitura com o original (nunca modifica o arquivo)
original = sqlite3.connect(f"file:{DB_ORIGINAL}?mode=ro", uri=True)
original.row_factory = sqlite3.Row

# As sprints existem só a partir de 2021? (evidência no original)
pd.read_sql_query("""\
    SELECT MIN(year) AS primeiro_ano,
           MAX(year) AS ultimo_ano,
           COUNT(*) AS corridas_com_sprint
    FROM race
    WHERE id IN (
        SELECT DISTINCT race_id FROM race_data WHERE type LIKE 'SPRINT%'
    )
""", original)

,primeiro_ano,ultimo_ano,corridas_com_sprint
0,2021,2026,29


In [22]:
# Colunas relacionadas a sprint no original: nulos nas tabelas race e race_data
linhas = []
for tabela in ("race", "race_data"):
    total = original.execute(f"SELECT COUNT(*) FROM {tabela}").fetchone()[0]
    for row in original.execute(f"PRAGMA table_info({tabela})"):
        col = row[1]
        if "sprint" not in col.lower():
            continue
        nulos = original.execute(
            f"SELECT COUNT(*) FROM {tabela} WHERE {col} IS NULL"
        ).fetchone()[0]
        linhas.append(
            {
                "table": tabela,
                "column": col,
                "nulls": nulos,
                "rows": total,
                "null_pct": round(100 * nulos / total, 2),
            }
        )

colunas_sprint = pd.DataFrame(linhas).sort_values("null_pct", ascending=False)
original.close()

colunas_sprint

,table,column,nulls,rows,null_pct
7,race,sprint_race_scheduled_laps,1168,1172,99.66
8,race,sprint_race_scheduled_distance,1168,1172,99.66
4,race,sprint_race_time,1154,1172,98.46
1,race,sprint_qualifying_date,1154,1172,98.46
2,race,sprint_qualifying_time,1154,1172,98.46
3,race,sprint_race_date,1154,1172,98.46
0,race,sprint_qualifying_format,1148,1172,97.95
6,race,sprint_race_distance,1142,1172,97.44
5,race,sprint_race_laps,1142,1172,97.44


## 4. Limpeza das corridas sprint

Com o embasamento acima (formato novo, apenas pós-2021, outro ruleset e nulos demais),
as sprints não ajudam a prever o standing da corrida normal. Removemos então da base de
trabalho tudo que é relacionado a sprint:

- as corridas com sessões sprint em `race_data`;
- também as corridas que têm metadados de sprint preenchidos na própria `race`
  (ex.: corridas futuras do calendário que ainda não têm dados de sessão, como
  o GP de Cingapura 2026);
- os registros de sessão sprint em `race_data`.

In [23]:
# Remove as corridas sprint da tabela race:
# por sessão sprint em race_data OU por metadados de sprint na própria race
deletadas_race = run("""\
    DELETE FROM race
    WHERE id IN (
        SELECT DISTINCT race_id
        FROM race_data
        WHERE type LIKE 'SPRINT%'
    )
    OR COALESCE(
        sprint_qualifying_format,
        sprint_qualifying_date,
        sprint_qualifying_time,
        sprint_race_date,
        sprint_race_time,
        sprint_race_laps,
        sprint_race_distance,
        sprint_race_scheduled_laps,
        sprint_race_scheduled_distance
    ) IS NOT NULL
""")

# Remove os registros de sessão sprint da tabela race_data
deletadas_race_data = run("DELETE FROM race_data WHERE type LIKE 'SPRINT%'")

print(f"Corridas sprint removidas de 'race': {deletadas_race}")
print(f"Registros de sessão sprint removidos de 'race_data': {deletadas_race_data}")
print(f"Corridas restantes: {row_count('race')}")

Corridas sprint removidas de 'race': 30
Registros de sessão sprint removidos de 'race_data': 1645
Corridas restantes: 1142


### Verificação da limpeza

Rodamos o relatório de nulos de novo para confirmar que as sprints realmente saíram. Os registros de sessão sprint em `race_data` devem
ser zero e as colunas de sprint em `race` devem ficar 100% nulas.

In [24]:
# 1) Nenhum registro de sessão sprint deve sobrar em race_data
query("SELECT COUNT(*) AS sprints_restantes_em_race_data FROM race_data WHERE type LIKE 'SPRINT%'")

,sprints_restantes_em_race_data
0,0


In [25]:
# 2) Null report de novo: colunas de sprint em race agora devem estar 100% nulas
report = full_null_report()

report[
    (report["table"] == "race")
    & report.column.str.contains("sprint", case=False)
][["column", "nulls", "rows", "null_pct"]].sort_values("null_pct", ascending=False)

,column,nulls,rows,null_pct
140,sprint_qualifying_format,1142,1142,100.0
169,sprint_qualifying_date,1142,1142,100.0
170,sprint_qualifying_time,1142,1142,100.0
171,sprint_race_date,1142,1142,100.0
172,sprint_race_time,1142,1142,100.0
173,sprint_race_laps,1142,1142,100.0
174,sprint_race_distance,1142,1142,100.0
175,sprint_race_scheduled_laps,1142,1142,100.0
176,sprint_race_scheduled_distance,1142,1142,100.0


## 5. Exploração de `race_data`

Nesta seção entendemos as sessões registradas em `race_data`.
Como as corridas sprint foram removidas da base de trabalho, os dados refletem
apenas o fim de semana convencional de corrida, que é o que interessa para a previsão dos standings.

### Tipos de sessão em `race_data`

Contagem de linhas por tipo de sessão.

In [26]:
query("""\
    SELECT type, COUNT(type) AS n
    FROM race_data
    GROUP BY type
    ORDER BY n DESC
""")

,type,n
0,RACE_RESULT,27577
1,QUALIFYING_RESULT,26998
2,STARTING_GRID_POSITION,25814
3,PIT_STOP,22490
4,FASTEST_LAP,17126
5,FREE_PRACTICE_1_RESULT,16120
6,FREE_PRACTICE_2_RESULT,15531
7,FREE_PRACTICE_3_RESULT,8845
8,QUALIFYING_1_RESULT,7707
9,WARMING_UP_RESULT,7683
